In [1]:
import pandas as pd 
import numpy as np 
from faker import Faker 
from datetime import timedelta 
import random

fake = Faker()
np.random.seed(42)
random.seed(42)

In [2]:
roles = ["Technician", "Planner", "QualityInspector", "Maintenance", "Worker", "SafetyOfficer"]
sources = ["LinkedIn", "Referral", "University", "Agency", "Internal", "JobFair"]
locations = ["Istanbul", "Ankara", "Izmir", "Bursa", "Adana", "Antalya"]
salary_bands = ["L1", "L2", "L3", "L4", "L5"]
stage_exits = ["ScreenFail", "InterviewFail", "OfferDeclined", "NoShow", "Withdrawn", "Hired"]

start_date = pd.Timestamp("2023-09-01")
end_date = pd.Timestamp("2024-03-01")

In [3]:
records = []

for i in range(900):
    candidate_id = f"C{i+1:04d}"
    role = random.choice(roles)
    source = random.choices(sources, weights=[30, 15, 20, 20, 10, 5])[0]
    location = random.choice(locations)
    salary_band = random.choice(salary_bands)
    
    # Son 6 ayda başvuruysa gecikme daha fazla
    applied_date = start_date + timedelta(days=random.randint(0, 540))
    is_recent = applied_date >= pd.Timestamp('2024-09-01')
    
    # Her aşama arası geçen gün — son 6 ayda daha uzun
    delay_multiplier = 2.0 if is_recent else 1.0
    
    d_screen = random.randint(3, int(10 * delay_multiplier))
    d_interview = random.randint(5, int(20 * delay_multiplier))
    d_offer = random.randint(3, int(10 * delay_multiplier))
    d_start = random.randint(7, int(21 * delay_multiplier))
    
    screen_date = applied_date + timedelta(days=d_screen)
    
    # Aday eleme olasılıkları — Agency'de daha kötü
    if source == 'Agency':
        exit_weights = [25, 20, 25, 10, 10, 10]  # OfferDeclined yüksek
    else:
        exit_weights = [20, 20, 10, 8, 7, 35]  # Hired daha yüksek
    
    stage_exit = random.choices(stage_exits, weights=exit_weights)[0]
    
    # Tarihleri stage_exit'e göre doldur
    if stage_exit == 'ScreenFail':
        interview_date = offer_date = start_date_emp = None
    elif stage_exit == 'InterviewFail':
        interview_date = screen_date + timedelta(days=d_interview)
        offer_date = start_date_emp = None
    elif stage_exit in ['OfferDeclined', 'NoShow', 'Withdrawn']:
        interview_date = screen_date + timedelta(days=d_interview)
        offer_date = interview_date + timedelta(days=d_offer)
        start_date_emp = None
    else:  # Hired
        interview_date = screen_date + timedelta(days=d_interview)
        offer_date = interview_date + timedelta(days=d_offer)
        start_date_emp = offer_date + timedelta(days=d_start)
    
    records.append({
        'Candidate_ID': candidate_id,
        'Role': role,
        'Source': source,
        'Location': location,
        'Salary_Band': salary_band,
        'Applied_Date': applied_date,
        'Screen_Date': screen_date,
        'Interview_Date': interview_date,
        'Offer_Date': offer_date,
        'Start_Date': start_date_emp,
        'Stage_Exit': stage_exit,
        'Is_Recent': is_recent
    })

recruitment_df = pd.DataFrame(records)
print(recruitment_df.shape)
print(recruitment_df['Stage_Exit'].value_counts())

(900, 12)
Stage_Exit
Hired            261
ScreenFail       203
InterviewFail    160
OfferDeclined    124
NoShow            84
Withdrawn         68
Name: count, dtype: int64


In [4]:
recruitment_df.groupby('Source')['Stage_Exit'].value_counts(normalize=True).unstack().round(2)

Stage_Exit,Hired,InterviewFail,NoShow,OfferDeclined,ScreenFail,Withdrawn
Source,,,,,,
Agency,0.08,0.16,0.09,0.30,0.28,0.11
Internal,0.29,0.20,0.09,0.17,0.20,0.05
JobFair,0.29,0.13,0.16,0.11,0.24,0.07
LinkedIn,0.36,0.18,0.10,0.07,0.21,0.08
Referral,0.37,0.19,0.08,0.08,0.23,0.05
University,0.35,0.18,0.08,0.11,0.20,0.07


In [5]:
recruitment_df.to_csv('../data/recruitment.csv', index=False)
print("Kaydedildi.")

Kaydedildi.


In [6]:
departments = ['LineMaintenance', 'BaseMaintenance', 'Planning', 'Quality', 'Safety', 'Operations']
managers = [f"M{i:03d}" for i in range(1, 16)]  # 15 farklı yönetici
cert_levels = ['None', 'Basic', 'Advanced']

employees = []

for i in range(350):
    emp_id = f"E{i+1:04d}"
    role = random.choice(roles)
    department = random.choice(departments)
    manager_id = random.choice(managers)
    
    emp_start = start_date + timedelta(days=random.randint(0, 500))
    tenure_days = (pd.Timestamp('2025-03-01') - emp_start).days # görev süresi
    
    training_completion = random.choices([0, 1], weights=[30, 70])[0]
    cert_level = random.choice(cert_levels)
    
    # Bazı yöneticilerde overtime daha yüksek
    if manager_id in ['M001', 'M002', 'M003']:
        overtime = round(random.uniform(60, 120), 1)  # yüksek overtime
    else:
        overtime = round(random.uniform(5, 50), 1)
    
    # Early attrition mantığı
    # Training düşük + overtime yüksek = daha yüksek çıkış riski
    attrition_risk = 0.15
    if training_completion == 0:
        attrition_risk += 0.20
    if overtime > 60:
        attrition_risk += 0.15
    
    exited = random.random() < attrition_risk # random.random 0-1 arasında değer döndürür
    
    if exited and tenure_days > 30:
        exit_days = random.randint(30, min(180, tenure_days))
        exit_date = emp_start + timedelta(days=exit_days)
        exit_reason = random.choices(
            ['Voluntary', 'Involuntary', 'Transfer'],
            weights=[60, 25, 15]
        )[0]
    else:
        exit_date = None
        exit_reason = None
    
    employees.append({
        'Employee_ID': emp_id,
        'Role': role,
        'Department': department,
        'Manager_ID': manager_id,
        'Start_Date': emp_start,
        'Exit_Date': exit_date,
        'Exit_Reason': exit_reason,
        'Training_Completion': training_completion,
        'Cert_Level': cert_level,
        'Overtime_Hours_6M': overtime,
        'Tenure_Days': tenure_days
    })

employee_df = pd.DataFrame(employees)
print(employee_df.shape)
print(employee_df['Exit_Reason'].value_counts(dropna=False))

(350, 11)
Exit_Reason
None           258
Voluntary       59
Involuntary     23
Transfer        10
Name: count, dtype: int64


In [7]:
employee_df.to_csv('../data/employee.csv', index=False)
print("Kaydedildi.")

Kaydedildi.
